In [0]:
# Only the pseudonymization SP can run this: it alone has the vault tables, the decrypt UDF and the keys.
# Proves that every published pseudonym can be reversed to the original (normalized) value.
# Prints counts only, so no cleartext ends up in job output.
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
catalog = dbutils.widgets.get("catalog")
RAW, VAULT, ANALYTICS = f"{catalog}.pii_raw", f"{catalog}.pii_vault", f"{catalog}.pii_analytics"

result = spark.sql(f"""
WITH reid AS (
  SELECT a.customer_id, a._batch_id,
         e.value AS email, n.value AS full_name,
         {VAULT}.pii_decrypt(a.ssn_enc,   'ssn_key_v1')   AS ssn,
         {VAULT}.pii_decrypt(a.phone_enc, 'phone_key_v1') AS phone,
         {VAULT}.pii_decrypt(a.dob_enc,   'dob_key_v1')   AS dob,
         {VAULT}.pii_hmac({VAULT}.pii_decrypt(a.ssn_enc, 'ssn_key_v1')) = a.ssn_hmac AS hmac_ok
  FROM {ANALYTICS}.customers a
  LEFT JOIN {VAULT}.tokmap_email e ON a.email_token = e.token
  LEFT JOIN {VAULT}.tokmap_name  n ON a.full_name_token = n.token
)
SELECT count(*) AS published_rows,
       count_if(r.email = v.email)                          AS email_matches,
       count_if(r.full_name = v.full_name)                  AS name_matches,
       count_if(r.ssn = v.ssn)                              AS ssn_matches,
       count_if(r.phone = v.phone)                          AS phone_matches,
       count_if(r.dob <=> CAST(v.dob AS STRING))            AS dob_matches,
       count_if(r.hmac_ok)                                  AS hmac_matches
FROM reid r
JOIN {RAW}.customers_valid v ON r.customer_id = v.customer_id AND r._batch_id = v._batch_id
""").first().asDict()
print(result)
assert result["published_rows"] > 0, "nothing published"
assert all(v == result["published_rows"] for v in result.values()), f"re-identification mismatch: {result}"